# where-clip-negative — worked example 3: Two-sided clamp with stacked where

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `where-clip-negative`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A two-sided clamp to `[lo, hi]` stacks two `t.where` calls: first lift values below `lo` up to `lo`, then pull values above `hi` down to `hi`. A precondition check raises `ValueError` when `lo > hi`. The result matches `torch.clamp` but is built from `where` primitives.

## Worked solution

We clamp a tensor into `[lo, hi]` using two selections.

1. We validate `lo <= hi` and raise `ValueError` otherwise — an impossible interval is a caller error, not something to silently fix.
2. First `where`: `t.where(x < lo, full_like(x, lo), x)` lifts under-floor values to `lo`.
3. Second `where`: applied to that result, `t.where(y > hi, full_like(y, hi), y)` pulls over-ceiling values to `hi`.
4. `full_like` builds a constant tensor of the bound, matching shape and dtype.

We print a spread of values clamped to `[-1, 1]`.

In [ ]:
import torch as t

def clamp_via_where(x: t.Tensor, lo: float, hi: float) -> t.Tensor:
    if lo > hi:
        raise ValueError(f'lo must be <= hi, got lo={lo} hi={hi}')
    y = t.where(x < lo, t.full_like(x, lo), x)
    y = t.where(y > hi, t.full_like(y, hi), y)
    return y

x = t.tensor([-3.0, -0.5, 0.2, 1.5, 4.0])
print('clamped:', clamp_via_where(x, -1.0, 1.0).tolist())